### Notebook 3 — Train / Validation / Test Split

### Step 1 — Load the labeled table

We load the labeled table created in the previous notebook. This table will be split into training, validation, and test sets before performing any deep analysis.


In [1]:
import pandas as pd

ml_table = pd.read_csv("../artifacts/labeled_table.csv")

print("Labeled table shape:", ml_table.shape)
print("Columns:")
print(ml_table.columns.tolist())

Labeled table shape: (96476, 20)
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'payment_total', 'payment_count', 'late']


### Step 2 — Check the date range

We check the range of order purchase dates to understand the time span of the dataset before choosing the splitting strategy.


In [2]:
# Check the date range

ml_table["order_purchase_timestamp"] = pd.to_datetime(
    ml_table["order_purchase_timestamp"]
)

print("Earliest order:", ml_table["order_purchase_timestamp"].min())
print("Latest order:", ml_table["order_purchase_timestamp"].max())

Earliest order: 2016-09-15 12:16:38
Latest order: 2018-08-29 15:00:37


### Step 3 — Choose the splitting strategy

Because the dataset contains orders over a two-year period, a time-based split is more representative of a real prediction scenario.

We will use earlier orders for training, followed by later orders for validation and testing. This prevents future data from being used to train the model.


### Step 4 — Check label balance over time

Before creating the time-based split, we check how the late-delivery rate changes over time. This helps us choose suitable boundaries for the training, validation, and test periods.

We check the number and percentage of late orders across years to understand whether the target distribution changes over time.

**Column meanings:**
- order_year → Order year
- total_orders → Total orders per year
- late_orders → Late orders per year
- late_percentage → Percentage of late orders 

In [5]:
# Check late-delivery rate by year

ml_table["order_year"] = ml_table["order_purchase_timestamp"].dt.year

yearly_label_balance = (
    ml_table.groupby("order_year")["late"]
    .agg(total_orders="count", late_orders="sum")
    .reset_index()
)

yearly_label_balance["late_percentage"] = (
    yearly_label_balance["late_orders"] / yearly_label_balance["total_orders"] * 100
).round(2)

yearly_label_balance

,order_year,total_orders,late_orders,late_percentage
0,2016,272,4,1.47
1,2017,43426,2878,6.63
2,2018,52778,4945,9.37


**Result:** 
Late deliveries increased over time: 
1.47% (2016) → 6.63% (2017) → 9.37% (2018).

This supports using a time-based split, as the target distribution changes over time.

### Step 5 — Split the Data

We split the data chronologically into training, validation, and test sets using a 70/15/15 ratio.

This ratio provides enough data for training while keeping sufficient data for validation and testing. The split is time-based, so earlier orders are used for training and later orders for validation and testing.

In [6]:
# Sort orders chronologically
ml_table = ml_table.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Calculate split points
n = len(ml_table)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Create time-based splits
train = ml_table.iloc[:train_end].copy()
validation = ml_table.iloc[train_end:val_end].copy()
test = ml_table.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 21)
Validation: (14471, 21)
Test: (14472, 21)


### Step 6 — Check the Splits

We check the date range and late-delivery percentage in each split to verify that the chronological split was applied correctly.


In [7]:
# Check date range and label balance for each split

for name, df in {"Train": train, "Validation": validation, "Test": test}.items():
    late_percentage = df["late"].mean() * 100

    print(f"\n{name}")
    print("Rows:", len(df))
    print(
        "Date range:",
        df["order_purchase_timestamp"].min(),
        "to",
        df["order_purchase_timestamp"].max(),
    )
    print("Late percentage:", round(late_percentage, 2), "%")


Train
Rows: 67533
Date range: 2016-09-15 12:16:38 to 2018-04-15 20:07:56
Late percentage: 9.03 %

Validation
Rows: 14471
Date range: 2018-04-15 20:10:23 to 2018-06-21 07:50:39
Late percentage: 5.34 %

Test
Rows: 14472
Date range: 2018-06-21 08:29:29 to 2018-08-29 15:00:37
Late percentage: 6.61 %


### Step 7 — Save the Splits

We save the training, validation, and test sets as separate artifacts for use in the following notebooks.

In [8]:
train.to_csv("../artifacts/train.csv", index=False)
validation.to_csv("../artifacts/validation.csv", index=False)
test.to_csv("../artifacts/test.csv", index=False)

print("Train, validation, and test sets saved successfully.")

Train, validation, and test sets saved successfully.


### Step 8 — Final Verification

We verify the number of rows and ensure that the order IDs do not overlap between the three splits.


In [9]:
print("Train rows:", len(train))
print("Validation rows:", len(validation))
print("Test rows:", len(test))

print("\nTotal rows:", len(train) + len(validation) + len(test))

print("\nOverlapping orders:")
print("Train & Validation:", len(set(train["order_id"]) & set(validation["order_id"])))
print("Train & Test:", len(set(train["order_id"]) & set(test["order_id"])))
print("Validation & Test:", len(set(validation["order_id"]) & set(test["order_id"])))

Train rows: 67533
Validation rows: 14471
Test rows: 14472

Total rows: 96476

Overlapping orders:
Train & Validation: 0
Train & Test: 0
Validation & Test: 0
